# SFMTL Server

> Paper Implementation: http://proceedings.mlr.press/v162/chen22a/chen22a.pdf


In [ ]:
#| default_exp servers.sfmtl

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import os
import pickle
import random
import copy
from tqdm import tqdm
import numpy as np
import pandas as pd
from loguru import logger
from collections import defaultdict
import networkx as nx
from community import community_louvain

import torch
import torch.nn.functional as F

from fedai.servers.base_server import BaseServer
from fedai.utils.registery import AlgorithmRegistry

## SFMTL Server

In [ ]:
#| export
@AlgorithmRegistry.register_server("sfmtl")
class SFMTLServer(BaseServer):
    def __init__(self,
                 cfg,
                 client_selector,
                 client_cls,
                 criterion,
                 fds,
                 writer= None,
                 device= None,
                 **kwargs
                 ):  
                 
        super().__init__(cfg, client_selector, client_cls, criterion, fds, writer, device, **kwargs)
        self.classes = self.cfg.data.classes
        self.idx_to_cls = {i: self.classes[i] for i in range(len(self.classes))}

### Server-Side logic

At the server, we are doing the following steps:
- Form the coalitions.
  - First, construct the weighted undirected graph $\mathcal{g}$.
  - Second, pass this graph to the louvian algorithm to get the communities.
- aggregate based on the equations stated in the next cells.


First, model similrity is compared using the norm of the difference between the two models as follows:
$$\operatorname{Sim}_{\text {head }}(\bm{w}_k,\bm{w}_\ell)=\left\|\bm{w}_k-\bm{w}_\ell\right\|,$$


In [ ]:
#| export
@patch
def model_similarity(self: SFMTLServer, h1, h2, model1, model2):

    # model_cls = get_cls("fedai.vision.models", self.cfg.model.name)

    # m1 = model_cls().head
    # m2 = model_cls().head  # Use head for both models

    m1 = copy.deepcopy(self.model.head)
    m2 = copy.deepcopy(self.model.head)
    with torch.no_grad():
        for k, v in model1.items():
            if k.startswith("head."):
                sub_k = k[len("head."):]
                if sub_k in m1.state_dict():
                    self.logger.info(f"copying {sub_k} to m1")
                    m1.state_dict()[sub_k].copy_(v)
                else:
                    self.logger.info(f"Key {sub_k} not found in m1.state_dict()")

        for k, v in model2.items():
            if k.startswith("head."):
                sub_k = k[len("head."):]
                if sub_k in m2.state_dict():
                    self.logger.info(f"copying {sub_k} to m2")
                    m2.state_dict()[sub_k].copy_(v)
                else:
                    self.logger.info(f"Key {sub_k} not found in m2.state_dict()")


    m1.to(self.device)
    m2.to(self.device)
    
    avg_sim = 0.0
    for h in [h1, h2]:
        h = h.to(self.device)
        with torch.no_grad():
            out1 = m1(h)
            out2 = m2(h)
            sim = F.cosine_similarity(out1, out2, dim=1).mean()
            avg_sim += sim.item()

    return avg_sim / 2  # correct normalization


We compare the embedding closeness using cosine similiairy.


$$\operatorname{Sim}_{\mathrm{repr}}(k, \ell) = \cos(\Omega) = \frac{\mathbf{h}_k \cdot \mathbf{h}_{\ell}}{\|\mathbf{h}_k\| \|\mathbf{h}_{\ell}\|}$$

In [ ]:
#| export
@patch
def h_similarity(self: SFMTLServer, h1, h2, label_set, label_set2):
    h1 = h1.reshape(self.cfg.data.num_classes, self.cfg.model.hidden_dim)
    h2 = h2.reshape(self.cfg.data.num_classes, self.cfg.model.hidden_dim)
    
    h1_norm = F.normalize(h1, p=2, dim=1)  # (3, 512)
    h2_norm = F.normalize(h2, p=2, dim=1)  # (3, 512)
    
    cos_sim_matrix = torch.mm(h1_norm, h2_norm.T)  # (10, 10)
    max_similarity = cos_sim_matrix.max(dim=1).values.mean()# This gives a higher similarity score if each class in h1 has at least one good match in h2.
    data = cos_sim_matrix.cpu().numpy()

    # index only data for the label_set
    data = data[label_set][:, label_set2]

    cols1 = [self.idx_to_cls[i] for i in label_set]
    cols2 = [self.idx_to_cls[i] for i in label_set2]

    df = pd.DataFrame(data, columns= cols1, index= cols2)
    return df, max_similarity.item()

We build a similarity graph using both the local represntation $h$ and the classification head $w$ where similairty is defined as:
$$\operatorname{Sim} = \sum_{i \in S} \alpha \cdot \operatorname{Sim}_{\text {head }}(C_j)+(1-\alpha) \cdot \operatorname{Sim}_{\mathrm{repr}}(C_j)$$

In [ ]:
#| export
@patch
def build_graph(self: SFMTLServer, lst_active_ids, comm_round):

    num_active = len(lst_active_ids)
    graph = np.zeros((num_active, num_active))

    clients_sim_dict = {}
    visited = {}
    for i, id in enumerate(lst_active_ids):
        state = self.state_mgr.get_state(id)
        model1 = state['model']
        h1 = state['h']
        label_set = state['label_set']

        for j, other_id in enumerate(lst_active_ids):
            if i == j or (id, other_id) in visited:
                continue
            other_state = self.state_mgr.get_state(other_id)
            model2 = other_state['model']
            h2 = other_state['h']
            label_set2 = other_state['label_set']

            w_sim = self.model_similarity(h1, h2, model1, model2)
            h_sim_df, h_sim = self.h_similarity(h1, h2, label_set, label_set2)
            clients_sim_dict[(id, other_id)] = h_sim_df
            
            val = (self.cfg.alpha) * w_sim + (1 - self.cfg.alpha) * h_sim
            self.logger.info(f"Client {id} and {other_id} similarity: {val}")
            self.logger.info(f"Client {id} and {other_id} h similarity: {h_sim}")
            self.logger.info(f"Client {id} and {other_id} w similarity: {w_sim}")
            val = round(val, 3)
            val = max(val, 0)  # prevent negative edge weights
            graph[i][j] = graph[j][i] = val

            visited[(id, other_id)] = True
            visited[(other_id, id)] = True

    self.logger.info(f"Before sym: {graph}")
    


    edges = []
    for i in range(num_active):
        for j in range(num_active):
            if i != j:
                edges.append((i, j, graph[i][j]))
                
    G = nx.Graph()
    G.add_weighted_edges_from(edges)

    for node, label in zip(list(range(num_active)), lst_active_ids):
        G.nodes[node]['label'] = label
    
    df_path = os.path.join(self.cfg.save_dir, str(comm_round), f"h_sim_df_{str(comm_round)}.pth")
    if not os.path.exists(os.path.dirname(df_path)):
        os.makedirs(os.path.dirname(df_path))
    torch.save(clients_sim_dict, df_path)

    return G, graph

In [ ]:
#| export
@patch    # save the model to self.cfg.save_dir/comm_round/f"local_output_{id}"/state.pth

def build_random_graph(self: SFMTLServer, lst_active_ids, comm_round):

    num_active = len(lst_active_ids)

    # seed the random number generator for reproducibility
    np.random.seed(42)
    clients_sim_dict = {}
    visited = {}


    G = nx.erdos_renyi_graph(num_active, 0.3)
    for (u, v) in G.edges():
        G.edges[u, v]['weight'] = round(random.uniform(0, 1), 3)

    zero_mat = np.zeros((num_active, num_active))

    for i in range(num_active):
        for j in range(num_active):
            if i != j and (i, j) not in visited:
                # checj if i and j are connected
                if G.has_edge(i, j):
                    zero_mat[i][j] = G.edges[i, j]['weight']
                visited[(i, j)] = True
                visited[(j, i)] = True
                

    for node, label in zip(list(range(num_active)), lst_active_ids):
        G.nodes[node]['label'] = label
    
    df_path = os.path.join(self.cfg.save_dir, str(comm_round), f"h_sim_df_{str(comm_round)}.pth")
    if not os.path.exists(os.path.dirname(df_path)):
        os.makedirs(os.path.dirname(df_path))
    torch.save(clients_sim_dict, df_path)

    return G, zero_mat

Now, we have a wighted graph, we nedd to form the coalitions (detecting the communities). We do so by using the louvain method for graph partitioning.

In [ ]:
#| export
@patch
def get_coalitions(self: SFMTLServer, G):
    correct_clients_indices = nx.get_node_attributes(G, 'label')
    partitions = community_louvain.best_partition(G)
    communities = defaultdict(list)
    for client, community in partitions.items():
        communities[community].append(client)
    communities = dict(communities)

    for community, clients in communities.items():
        communities[community] = [correct_clients_indices[client] for client in clients]

    return communities


### Aggregation

The aggregation rule here is two-folds:
- Representations are aggregated as:
  $$ h_c = \sum_{k \in C_{j}} \frac{\zeta_k}{\sum_{k \in C_{j}} \zeta_k}h_k$$
  where $\zeta$ is the shapely value.
  
The Classification heads are aggregated as:
$$  w_k^{(t+1)} = w_{k, R}^{(t)} - \lambda \eta_2 \sum_{\ell \in C_{j}} a_{k, \ell} (w_{k,R}^{(t)} - w_{\ell, R}^{(t)})$$

In [ ]:
#| export
@patch
def aggregate(self: SFMTLServer, lst_active_ids, comm_round, len_clients_ds, save_to_disk= False):

    self.graph, self.akl_connection = self.build_graph(lst_active_ids, comm_round) \
    if self.cfg.graph_type == "similarity" else self.build_random_graph(lst_active_ids, comm_round)

    graph_path = os.path.join(self.cfg.save_dir, str(comm_round), f"graph_{str(comm_round)}.gpickle")
    with open(graph_path, "wb") as f:
        pickle.dump(self.graph, f, pickle.HIGHEST_PROTOCOL)

    self.coalitions = self.get_coalitions(self.graph)
    coalitions_path = os.path.join(self.cfg.save_dir, str(comm_round), "coalitions.pth")
    torch.save(self.coalitions, coalitions_path)

    global_lr = float(self.cfg.optimizer.lr) * float(self.cfg.local_epochs)
    reg_param = self.cfg.lambda_
    
    with torch.no_grad():
        coalitions_reprs = {}
        for col_ind, lst_clients in self.coalitions.items():

            m_t = sum(len_clients_ds[id] for id in lst_clients)
            for i, id in enumerate(lst_clients):
                if not id in lst_active_ids:
                    continue
                # state_path = os.path.join(self.cfg.save_dir, str(comm_round), f"local_output_{id}", "state.pth")
                # state = torch.load(state_path, weights_only= False)
                state = self.state_mgr.get_state(id)
                client_h = state['h']

                if i == 0:
                    col_repr = torch.zeros_like(client_h)

                n_k = len_clients_ds[id]
                weight =  n_k / m_t 

                col_repr.add_(weight * client_h)
            coalitions_reprs[col_ind] = col_repr
            

        aggregated_states = []
        for col_ind, lst_clients in self.coalitions.items():
            for i, id in enumerate(lst_clients):
                if not id in lst_active_ids:
                    continue
                # state_path = os.path.join(self.cfg.save_dir, str(comm_round), f"local_output_{id}", "state.pth")
                # state = torch.load(state_path, weights_only= False)
                
                state = self.state_mgr.get_state(id)
                client_model = state['model']

                client_diff = {
                    key: torch.zeros_like(value) 
                    # for key, value in client_model.items() if key.startswith("fc2") or key.startswith("dropout")
                    for key, value in client_model.items() if key.startswith("head")
                }

                for j, other_id in enumerate(lst_clients):
                    if i == j:
                        continue
                    # other_state_path = os.path.join(self.cfg.save_dir, str(comm_round), f"local_output_{other_id}", "state.pth")
                    # other_state = torch.load(other_state_path, weights_only= False)

                    other_state = self.state_mgr.get_state(other_id)
                    other_client_model = other_state['model']

                    a_kl = self.akl_connection[i, j]
                    for key in client_model.keys():
                        # if key.startswith("fc2") or key.startswith("dropout"):
                        if key.startswith("head"):
                            client_diff[key].add_(a_kl * (client_model[key] - other_client_model[key]))

                for key in client_model.keys():
                    if key.startswith("head"):
                        client_model[key].sub_(global_lr * reg_param * client_diff[key])

                clinet_state = {
                    'model': client_model.state_dict(),
                    'optimizer': state['optimizer'],
                    'h': state['h'],
                    'h_c': coalitions_reprs[col_ind],
                }
                aggregated_states.append(clinet_state)

                if save_to_disk:
                    agg_client_state_path = os.path.join(self.cfg.save_dir, str(comm_round), f"aggregated_model_{id}", "state.pth")
                    
                    if not os.path.exists(os.path.dirname(agg_client_state_path)):
                        os.makedirs(os.path.dirname(agg_client_state_path))

                    torch.save(clinet_state, agg_client_state_path)

        for i, id in enumerate(lst_active_ids):
            self.state_mgr.set_state(id, aggregated_states[i])   

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()